In [1]:
import fugashi
from collections import Counter
import math
import os

def calculate_tf_idf_for_nouns(file_path="kokoro.txt", top_n=20):
    """
    指定されたテキストファイル内の名詞のTF-IDFスコアを計算し、
    上位N語とそのTF, IDF, TF-IDFスコアを表示します。
    """
    if not os.path.exists(file_path):
        print(f"エラー: ファイル '{file_path}' が見つかりません。")
        print("ファイルパスが正しいか、ファイルが同じディレクトリにあるか確認してください。")
        return

    tagger = fugashi.Tagger()

    all_nouns = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # 形態素解析と名詞の抽出
        for word in tagger(text):
            if word.pos.startswith('名詞'):
                all_nouns.append(word.surface)

        # 1. TF (Term Frequency) の計算
        # 各名詞の出現回数
        noun_counts = Counter(all_nouns)
        total_nouns = len(all_nouns) # 文書内の全名詞数

        tf_scores = {}
        for noun, count in noun_counts.items():
            tf_scores[noun] = count / total_nouns

        # 2. IDF (Inverse Document Frequency) の計算
        # 今回は単一の文書なので、各単語のIDFは log(1 / (文書内出現有無)) となる。
        # 単語が文書に出現すれば1、そうでなければ0になるが、IDFは対数を取るため、
        # log(全文書数 / その単語を含む文書数) で計算するのが一般的。
        # 単一文書の場合、全ての単語がその文書に含まれるため、IDFは log(1/1) = 0 になってしまう。
        # これでは単語の相対的な重みが出ないため、ここでは簡易的に「1 + log(全単語数 / その単語の出現数)」
        # のような形で、TFと組み合わせて単語の重要度を出す代替策を用いるか、
        # もしくは一般的なIDFの定義に従い、log(1 + 1) = 0.693 などのオフセットを加える。
        # ここでは、scikit-learnのTfidfVectorizerで用いられる慣例的なIDF計算式 (1 + log(N/df)) に倣い、
        # ドキュメント頻度dfを単語の出現有無(1)として、N=1 (ドキュメント数)と仮定するとIDFは0になるため、
        # ここでは単語の希少性を示すために、簡略化されたIDFの概念を適用します。
        # 例: log(総語数 / その語の出現頻度) を IDF の代替として用いる
        # または、より一般的なTF-IDFのライブラリの内部実装に準拠する。
        
        # 簡易的なIDFの計算 (単一文書向け): log(全名詞数 / その名詞の出現頻度)
        # この式はIDFの厳密な定義とは異なりますが、単語の相対的な希少性を表現できます。
        # ゼロ除算を避けるため +1 を加えます。
        idf_scores = {}
        for noun, count in noun_counts.items():
            # この計算方法は厳密なIDFではありませんが、単一文書内での単語の希少性を相対的に示します。
            # 例: tfidf = tf * (1 + log(N / df)) のようなIDFの部分
            idf_scores[noun] = math.log(total_nouns / (count + 1)) + 1 # +1 はゼロ除算対策と、値が小さくなりすぎないように

        # 3. TF-IDFスコアの計算
        tf_idf_scores = {}
        for noun in noun_counts.keys():
            tf_idf_scores[noun] = tf_scores[noun] * idf_scores[noun]

        print(f"--- '{file_path}' 内の名詞のTF-IDFスコア上位 {top_n} 語 ---")
        print("単語\t\tTF\t\tIDF\t\tTF-IDF")
        print("-" * 60)

        # TF-IDFスコアの高い順にN語を表示
        # (単語, スコア)のリストを作成し、スコアでソート
        sorted_tfidf = sorted(tf_idf_scores.items(), key=lambda item: item[1], reverse=True)

        for noun, tfidf in sorted_tfidf[:top_n]:
            tf = tf_scores[noun]
            idf = idf_scores[noun]
            print(f"{noun:<10}\t{tf:.6f}\t{idf:.6f}\t{tfidf:.6f}")

    except Exception as e:
        print(f"ファイル処理中にエラーが発生しました: {e}")

# 分析の実行
if __name__ == "__main__":
    calculate_tf_idf_for_nouns("kokoro.txt")

--- 'kokoro.txt' 内の名詞のTF-IDFスコア上位 20 語 ---
単語		TF		IDF		TF-IDF
------------------------------------------------------------
先生        	0.024768	4.696526	0.116323
事         	0.023894	4.732397	0.113075
Ｋ         	0.017109	5.065744	0.086668
奥         	0.016692	5.090315	0.084969
もの        	0.016443	5.105353	0.083945
時         	0.016151	5.123188	0.082746
父         	0.012322	5.393035	0.066450
自分        	0.010989	5.507037	0.060519
うち        	0.010032	5.597829	0.056158
一         	0.008409	5.773561	0.048548
方         	0.007992	5.824077	0.046548
母         	0.007659	5.866411	0.044933
気         	0.007451	5.893810	0.043916
人         	0.007368	5.904983	0.043508
前         	0.006993	5.956868	0.041658
嬢         	0.006993	5.956868	0.041658
今         	0.006452	6.036911	0.038951
中         	0.006244	6.069487	0.037898
上         	0.006078	6.096334	0.037051
顔         	0.005536	6.188927	0.034264
